<a href="https://colab.research.google.com/github/ever-oli/TensorPoly/blob/main/vit_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
install.packages("abind") # Install the `abind` package if not already installed
library(abind) # For efficient array stacking

patch_embed <- function(image, patch_size, embed_dim) {
  # Extract dimensions
  dim_image <- dim(image)
  batch <- dim_image[1]
  H <- dim_image[2]
  W <- dim_image[3]
  C <- dim_image[4]

  # Calculate number of patches
  num_patches_h <- floor(H / patch_size)
  num_patches_w <- floor(W / patch_size)
  num_patches <- num_patches_h * num_patches_w

  # --- Step 1 & 2: Extract and stack patches ---
  # The goal is to obtain a 6D array with dimensions:
  # (batch, num_patches_h, num_patches_w, patch_size, patch_size, C)

  all_patches_list <- list()

  for (b_idx in 1:batch) {
    batch_patches_list <- list()
    for (nh_idx in 1:num_patches_h) {
      row_start <- (nh_idx - 1) * patch_size + 1
      row_end <- nh_idx * patch_size

      for (nw_idx in 1:num_patches_w) {
        col_start <- (nw_idx - 1) * patch_size + 1
        col_end <- nw_idx * patch_size

        # Extract patch for the current batch, row block, and col block
        # In R, `image[b_idx, row_range, col_range, channel_range]` extracts a (patch_size, patch_size, C) array.
        current_patch <- image[b_idx, row_start:row_end, col_start:col_end, , drop = FALSE]

        # Store the 3D patch (patch_size, patch_size, C)
        batch_patches_list[[length(batch_patches_list) + 1]] <- current_patch
      }
    }

    # Reshape `batch_patches_list` into a 5D array for the current batch:
    # (num_patches_h, num_patches_w, patch_size, patch_size, C)
    batch_patches_array <- array(dim = c(num_patches_h, num_patches_w, patch_size, patch_size, C))

    idx = 1
    for (nh_idx in 1:num_patches_h) {
      for (nw_idx in 1:num_patches_w) {
        batch_patches_array[nh_idx, nw_idx, , , ] <- batch_patches_list[[idx]]
        idx = idx + 1
      }
    }
    all_patches_list[[b_idx]] <- batch_patches_array
  }

  # Combine all batch patches into a single 6D array using abind along a new first dimension (batch)
  # Resulting `patches` has dim (batch, num_patches_h, num_patches_w, patch_size, patch_size, C)
  patches <- do.call(abind, c(all_patches_list, list(along = 0)))

  # --- Step 3 & 4: Flatten each patch and reshape to sequence form ---
  # Target: (batch, num_patches, patch_dim)
  patch_dim <- patch_size * patch_size * C
  patches_seq <- array(dim = c(batch, num_patches, patch_dim))

  for (b_idx in 1:batch) {
    patch_idx_counter <- 1
    for (nh_idx in 1:num_patches_h) {
      for (nw_idx in 1:num_patches_w) {
        # Extract the 3D patch (patch_size, patch_size, C) for the current (b, nh, nw)
        # By default, R will 'drop' dimensions of size 1, making current_3d_patch 3D.
        current_3d_patch <- patches[b_idx, nh_idx, nw_idx, , , ]

        # To flatten (patch_size, patch_size, C) into a 1D vector (patch_dim)
        # in Python's row-major (channel-fastest) order:
        # 1. Transpose the patch dimensions to (C, patch_size, patch_size)
        #    (Original: rows, cols, channels. New: channels, cols, rows)
        # 2. Use as.vector, which flattens in column-major order (first dimension changes fastest).
        #    This will effectively flatten channels first, then columns, then rows, matching Python's default.
        flattened_patch <- as.vector(aperm(current_3d_patch, c(3, 2, 1)))

        # Assign the flattened patch to the correct position in patches_seq
        patches_seq[b_idx, patch_idx_counter, ] <- flattened_patch
        patch_idx_counter <- patch_idx_counter + 1
      }
    }
  }

  # --- Step 5: Linear projection to embedding dimension ---
  # Initialize projection weights (using stats::rnorm for random numbers)
  W_proj <- matrix(stats::rnorm(patch_dim * embed_dim), nrow = patch_dim, ncol = embed_dim) * 0.01

  # Apply linear projection (R's `%*%` operator handles matrix multiplication)
  # Explicitly handle batch dimension for matrix multiplication
  embeddings_list <- list()
  for (b_idx in 1:batch) {
    # Extract the 2D matrix for the current batch: (num_patches, patch_dim)
    current_batch_patches_matrix <- patches_seq[b_idx, , ]

    # Perform matrix multiplication: (num_patches, patch_dim) %*% (patch_dim, embed_dim) -> (num_patches, embed_dim)
    embeddings_list[[b_idx]] <- current_batch_patches_matrix %*% W_proj
  }

  # Combine results back into a 3D array: (batch, num_patches, embed_dim)
  if (batch == 1) {
    embeddings <- array(embeddings_list[[1]], dim = c(1, num_patches, embed_dim))
  } else {
    embeddings <- do.call(abind, c(embeddings_list, list(along = 0)))
  }

  return(embeddings)
}

# Example Usage:
# Load the `abind` library if not already loaded
# install.packages("abind") # Uncomment and run if abind is not installed
library(abind)

set.seed(123) # for reproducibility

# Create a dummy image array: batch, H, W, C
# e.g., 1 batch, 16x16 pixels, 3 channels (RGB)
img <- array(runif(1 * 16 * 16 * 3), dim = c(1, 16, 16, 3))

patch_s <- 4 # 4x4 patches
embed_d <- 768 # embedding dimension

output_embeddings <- patch_embed(img, patch_s, embed_d)

cat("Input image dimensions: ", paste(dim(img), collapse = ", "), "\n")
cat("Output embeddings dimensions: ", paste(dim(output_embeddings), collapse = ", "), "\n")
# Expected output for this example: (1, 16, 768)
# (batch, num_patches (16x16 / 4x4 = 16), embed_dim)


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



Input image dimensions:  1, 16, 16, 3 
Output embeddings dimensions:  1, 16, 768 


In [9]:
add_position_embedding <- function(patches, num_patches, embed_dim) {
  # Create learnable position embeddings (simulated with random initialization)
  # In practice, these would be learned during training
  position_embeddings <- array(stats::rnorm(1 * num_patches * embed_dim), dim = c(1, num_patches, embed_dim)) * 0.01

  # Add position embeddings to patch embeddings (R automatically handles broadcasting over the batch dimension)
  output <- patches + position_embeddings

  return(output)
}

# Example Usage (assuming patch_embed has been run and output_embeddings is available):
# set.seed(456) # for reproducibility
#
# # Assuming `output_embeddings` from the previous `patch_embed` function is available
# # For this example, let's create a dummy `patches` array if not available
# if (!exists("output_embeddings")) {
#   batch_size <- 1
#   num_patches_example <- 16 # (16x16 / 4x4) = 16 patches
#   embed_dim_example <- 768
#   patches_example <- array(runif(batch_size * num_patches_example * embed_dim_example),
#                          dim = c(batch_size, num_patches_example, embed_dim_example))
#   cat("Created dummy patches for example: ", paste(dim(patches_example), collapse = ", "), "\n")
#
#   pos_embeddings <- add_position_embedding(patches_example, num_patches_example, embed_dim_example)
#   cat("Output with position embeddings dimensions: ", paste(dim(pos_embeddings), collapse = ", "), "\n")
# } else {
#   # Use the output from the previous patch_embed function
#   patches_to_add_pos_embed <- output_embeddings
#   current_num_patches <- dim(patches_to_add_pos_embed)[2]
#   current_embed_dim <- dim(patches_to_add_pos_embed)[3]
#
#   pos_embeddings <- add_position_embedding(patches_to_add_pos_embed, current_num_patches, current_embed_dim)
#   cat("Output with position embeddings dimensions: ", paste(dim(pos_embeddings), collapse = ", "), "\n")
# }

In [10]:
library(abind) # For efficient array stacking

prepend_class_token <- function(patches, embed_dim) {
  batch_size <- dim(patches)[1]

  # Initialize learnable [CLS] token (simulated with random initialization)
  # Shape: (1, 1, embed_dim)
  cls_token_single <- array(stats::rnorm(embed_dim), dim = c(1, 1, embed_dim)) * 0.02

  # Broadcast [CLS] token to match batch size
  # Shape: (batch_size, 1, embed_dim)
  # Replicate the single token for each batch and stack them
  cls_token_batch_list <- lapply(1:batch_size, function(x) cls_token_single)
  cls_token_batch <- do.call(abind, c(cls_token_batch_list, list(along = 1)))

  # Concatenate [CLS] token at the beginning of the sequence (along dimension 2)
  output <- abind(cls_token_batch, patches, along = 2)

  return(output)
}

# Example Usage (assuming patch_embed has been run and output_embeddings is available):
set.seed(789) # for reproducibility

if (!exists("output_embeddings")) {
  batch_size_ex <- 1
  num_patches_ex <- 16
  embed_dim_ex <- 768
  patches_ex <- array(runif(batch_size_ex * num_patches_ex * embed_dim_ex),
                      dim = c(batch_size_ex, num_patches_ex, embed_dim_ex))
  cat("Created dummy patches for example: ", paste(dim(patches_ex), collapse = ", "), "\n")

  output_with_cls <- prepend_class_token(patches_ex, embed_dim_ex)
  cat("Output with CLS token dimensions (dummy): ", paste(dim(output_with_cls), collapse = ", "), "\n")
} else {
  patches_to_prepend <- output_embeddings
  current_embed_dim <- dim(patches_to_prepend)[3]

  output_with_cls <- prepend_class_token(patches_to_prepend, current_embed_dim)
  cat("Output with CLS token dimensions: ", paste(dim(output_with_cls), collapse = ", "), "\n")
}

Output with CLS token dimensions:  1, 17, 768 


In [12]:
install.packages("torch") # Install the torch package if not already installed
library(torch)

#' Layer normalization
layer_norm <- function(x, eps = 1e-6) {
  mean_val <- torch_mean(x, dim = -1, keepdim = TRUE)

  # unbiased = FALSE explicitly forces population variance, matching np.var
  var_val <- torch_var(x, dim = -1, unbiased = FALSE, keepdim = TRUE)

  return((x - mean_val) / torch_sqrt(var_val + eps))
}

#' GELU activation (approximation)
gelu <- function(x) {
  # Evaluates: 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))
  # torch handles standard R scalar broadcasting automatically
  scalar_term <- sqrt(2 / pi)
  inner_term <- x + 0.044715 * (x^3)

  return(0.5 * x * (1 + torch_tanh(scalar_term * inner_term)))
}

#' Softmax function
softmax <- function(x, dim = -1) {
  # torch_max returns a list: [[1]] is values, [[2]] is indices
  max_vals <- torch_max(x, dim = dim, keepdim = TRUE)[[1]]
  exp_x <- torch_exp(x - max_vals)

  return(exp_x / torch_sum(exp_x, dim = dim, keepdim = TRUE))
}

#' Multi-Head Self-Attention (simplified)
multi_head_self_attention <- function(x, num_heads, embed_dim) {
  dims <- x$shape
  batch <- dims[1]
  seq_len <- dims[2]
  head_dim <- as.integer(embed_dim / num_heads)

  device <- x$device

  # Simplified attention weights instantiated dynamically
  W_q <- torch_randn(c(embed_dim, embed_dim), device = device) * 0.02
  W_k <- torch_randn(c(embed_dim, embed_dim), device = device) * 0.02
  W_v <- torch_randn(c(embed_dim, embed_dim), device = device) * 0.02
  W_o <- torch_randn(c(embed_dim, embed_dim), device = device) * 0.02

  # Project to Q, K, V
  Q <- torch_matmul(x, W_q)
  K <- torch_matmul(x, W_k)
  V <- torch_matmul(x, W_v)

  # Reshape for multi-head and permute (equivalent to Python's transpose(0, 2, 1, 3))
  Q <- Q$view(c(batch, seq_len, num_heads, head_dim))$permute(c(1, 3, 2, 4))
  K <- K$view(c(batch, seq_len, num_heads, head_dim))$permute(c(1, 3, 2, 4))
  V <- V$view(c(batch, seq_len, num_heads, head_dim))$permute(c(1, 3, 2, 4))

  # Scaled dot-product attention
  # K is transposed on its final two dimensions
  scores <- torch_matmul(Q, K$transpose(3, 4)) / sqrt(head_dim)
  attn_weights <- softmax(scores, dim = -1)
  attn_output <- torch_matmul(attn_weights, V)

  # Reshape back to (batch, seq_len, embed_dim)
  attn_output <- attn_output$permute(c(1, 3, 2, 4))$contiguous()$view(c(batch, seq_len, embed_dim))

  # Output projection
  return(torch_matmul(attn_output, W_o))
}

#' MLP with GELU activation
mlp <- function(x, embed_dim, mlp_ratio) {
  hidden_dim <- as.integer(embed_dim * mlp_ratio)
  device <- x$device

  # Simplified MLP weights
  W1 <- torch_randn(c(embed_dim, hidden_dim), device = device) * 0.02
  b1 <- torch_zeros(c(hidden_dim), device = device)
  W2 <- torch_randn(c(hidden_dim, embed_dim), device = device) * 0.02
  b2 <- torch_zeros(c(embed_dim), device = device)

  # First linear layer + GELU
  h <- gelu(torch_matmul(x, W1) + b1)

  # Second linear layer
  return(torch_matmul(h, W2) + b2)
}

#' ViT Transformer encoder block (Pre-LayerNorm formulation)
vit_encoder_block <- function(x, embed_dim, num_heads, mlp_ratio = 4.0) {

  # =========== First sub-block: Multi-Head Self-Attention ===========
  x_norm1 <- layer_norm(x)
  attn_output <- multi_head_self_attention(x_norm1, num_heads, embed_dim)
  x <- x + attn_output

  # =========== Second sub-block: MLP ===========
  x_norm2 <- layer_norm(x)
  mlp_output <- mlp(x_norm2, embed_dim, mlp_ratio)
  x <- x + mlp_output

  return(x)
}

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘coro’, ‘safetensors’




In [14]:
classification_head <- function(encoder_output, num_classes) {
  # Step 1: Extract [CLS] token at position 0
  # In R, arrays are 1-indexed. The [CLS] token is at index 1.
  # Shape of cls_token_r_array: (batch, embed_dim)
  cls_token_r_array <- encoder_output[, 1, ]

  # R can drop dimensions if they are of size 1. If cls_token_r_array is a 1D vector (dim is NULL),
  # reshape it to (1, embed_dim).
  if (is.null(dim(cls_token_r_array))) {
    cls_token_r_array <- array(cls_token_r_array, dim = c(1, length(cls_token_r_array)))
  }

  # Step 2: Apply layer normalization to [CLS] token
  # Convert R array to torch_tensor for layer_norm function
  cls_token_tensor <- torch_tensor(cls_token_r_array)
  cls_norm_tensor <- layer_norm(cls_token_tensor) # Uses the torch-based layer_norm defined in another cell
  cls_norm_r_array <- as.array(cls_norm_tensor) # Convert back to R array for base R operations

  # Step 3: Linear projection to class logits
  # Initialize weights (would be learned in practice)
  embed_dim <- dim(cls_norm_r_array)[2] # Get embed_dim from the 2D array
  W <- matrix(stats::rnorm(embed_dim * num_classes), nrow = embed_dim, ncol = num_classes) * 0.01
  b <- rep(0, num_classes) # Bias vector

  # Project to class logits: (batch, embed_dim) %*% (embed_dim, num_classes) + (num_classes)
  logits <- cls_norm_r_array %*% W + b

  return(logits)
}

# Example Usage:
# Assuming output_with_cls from prepend_class_token is available in the environment
set.seed(999) # for reproducibility
num_classes_ex <- 10 # Example number of classes

if (!exists("output_with_cls")) {
  # Create a dummy encoder_output if not available from previous cells
  batch_size_ex <- 1
  seq_len_ex <- 17 # e.g., CLS token + 16 patches
  embed_dim_ex <- 768
  dummy_encoder_output <- array(runif(batch_size_ex * seq_len_ex * embed_dim_ex),
                                dim = c(batch_size_ex, seq_len_ex, embed_dim_ex))
  cat("Created dummy encoder_output for example: ", paste(dim(dummy_encoder_output), collapse = ", "), "\n")

  classification_logits <- classification_head(dummy_encoder_output, num_classes_ex)
  cat("Output classification logits dimensions (dummy): ", paste(dim(classification_logits), collapse = ", "), "\n")
} else {
  # Use the actual output from previous steps
  encoder_output_from_prev <- output_with_cls # Assuming output_with_cls is the input to the encoder block

  # A full ViT would have encoder_output as the result of vit_encoder_block(output_with_cls, ...)
  # For this example, we'll use output_with_cls directly if vit_encoder_block hasn't been run yet.
  # If you have executed vit_encoder_block, replace 'encoder_output_from_prev' with its output.

  classification_logits <- classification_head(encoder_output_from_prev, num_classes_ex)
  cat("Output classification logits dimensions: ", paste(dim(classification_logits), collapse = ", "), "\n")
}

Output classification logits dimensions:  1, 10 


In [15]:
library(torch)

#' Vision Transformer Structure Simulator
VisionTransformer <- nn_module(
  "VisionTransformer",

  initialize = function(image_size = 224, patch_size = 16,
                        num_classes = 1000, embed_dim = 768,
                        depth = 12, num_heads = 12, mlp_ratio = 4.0) {

    self$image_size <- image_size
    self$patch_size <- patch_size

    # Use %/% for strict integer division to match Python's //
    self$num_patches <- (image_size %/% patch_size)^2

    self$embed_dim <- embed_dim
    self$depth <- depth
    self$num_heads <- num_heads
    self$mlp_ratio <- mlp_ratio
    self$num_classes <- num_classes
  },

  forward = function(x) {
    # Extract batch size
    batch_size <- dim(x)[1]

    # Best practice: keep tensors on the same device as the input
    device <- x$device

    # 1. Patch embedding shape
    # Note: Like the Python script, this deliberately overwrites the input image tensor 'x'
    x_embed <- torch_zeros(c(batch_size, self$num_patches, self$embed_dim), device = device)

    # 2. Add class token
    # R is 1-indexed, so we concatenate along dimension 2 (the sequence length)
    cls_token <- torch_zeros(c(batch_size, 1, self$embed_dim), device = device)
    x_embed <- torch_cat(list(cls_token, x_embed), dim = 2)

    # 3. Add position embedding (shape preserved)
    pos_embed <- torch_zeros(c(1, self$num_patches + 1, self$embed_dim), device = device)
    x_embed <- x_embed + pos_embed

    # 4. Transformer blocks (shape preserved)
    for (i in 1:self$depth) {
      # Identity with correct shape
      x_embed <- x_embed + torch_zeros_like(x_embed)
    }

    # 5. Extract CLS token and classify
    # CRITICAL: R uses 1-based indexing. The CLS token is at index 1, not 0.
    cls_token_out <- x_embed[, 1, ]

    # Simulate logits
    logits <- torch_zeros(c(batch_size, self$num_classes), device = device)

    return(logits)
  }
)